# Baseline NLP Models for Sentiment Analysis

In this notebook, we establish a baseline for sentiment analysis using traditional machine learning algorithms (e.g., Logistic Regression, Naive Bayes, SVM) combined with TF-IDF vectorization. We will use the processed dataset `sentfin_strict.csv` and the Loughran-McDonald financial dictionary for initial classification context.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Load processed financial sentiment dataset
df = pd.read_csv('../data/processed/sentfin_strict.csv')
display(df.head())

,text,sentiment
0,SpiceJet to issue 6.4 crore warrants to promoters,neutral
1,MMTC Q2 net loss at Rs 10.4 crore,neutral
2,"Mid-cap funds can deliver more, stay put: Experts",positive
3,Mid caps now turn into market darlings,positive
4,"Market seeing patience, if not conviction: Pra...",neutral


## Setup: Imports, Data Prep & Results Store

We encode the sentiment labels, build a TF-IDF matrix and a 80/20 train-test split.
`model_results` will collect one dict per model with its name, best hyper-parameters, and full evaluation metrics.

In [6]:
!pip install xgboost
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, precision_score, recall_score
)
from xgboost import XGBClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
    --------------------------------------- 1.6/101.7 MB 12.0 MB/s eta 0:00:09
   - -------------------------------------- 3.7/101.7 MB 10.4 MB/s eta 0:00:10
   -- ------------------------------------- 5.2/101.7 MB 9.7 MB/s eta 0:00:10
   -- ------------------------------------- 5.8/101.7 MB 8.6 MB/s eta 0:00:12
   -- ------------------------------------- 6.3/101.7 MB 6.8 MB/s eta 0:00:15
   -- ------------------------------------- 6.8/101.7 MB 5.9 MB/s eta 0:00:17
   -- ------------------------------------- 7.6/101.7 MB 5.3 MB/s eta 0:00:18
   --- ------------------------------------ 8.1/101.7 MB 5.1 MB/s eta 0:00:19
   --- ------------------------------------ 8.9/101.7 MB 4.9 MB/s eta 0:00:19
   --- ------------------------------------ 10.0/101.7 MB 4.8 MB/s eta 0:00:20
   ---- ----------------------------------- 10.7/101.7 MB 4.8 MB/s eta 0:00:20
   ---- ----------------------------------- 11.5/101.7 MB 4.7 MB/s 

NameError: name 'X_train' is not defined

In [8]:
os.makedirs('../data/pkls', exist_ok=True)

# ── Load data ──────────────────────────────────────────────────────────────────
df = pd.read_csv('../data/processed/sentfin_strict.csv')
print(f"Dataset shape : {df.shape}")
print(f"Class distribution:\n{df['sentiment'].value_counts()}")

X = df['text'].astype(str)
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model_results = []

def evaluate_and_store(name, pipeline, best_params):
    """Evaluate a fitted pipeline and append results to model_results."""
    y_pred = pipeline.predict(X_test)
    report = classification_report(y_test, y_pred)
    entry = {
        "model"            : name,
        "best_params"      : best_params,
        "accuracy"         : accuracy_score(y_test, y_pred),
        "macro_f1"         : f1_score(y_test, y_pred, average='macro'),
        "weighted_f1"      : f1_score(y_test, y_pred, average='weighted'),
        "macro_precision"  : precision_score(y_test, y_pred, average='macro'),
        "macro_recall"     : recall_score(y_test, y_pred, average='macro'),
        "classification_report": report,
        "confusion_matrix" : confusion_matrix(y_test, y_pred),
    }
    model_results.append(entry)
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    print(f"  Best params : {best_params}")
    print(f"  Accuracy    : {entry['accuracy']:.4f}")
    print(f"  Macro F1    : {entry['macro_f1']:.4f}")
    print("\nClassification Report:")
    print(report)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("Setup complete.")


Dataset shape : (9518, 2)
Class distribution:
sentiment
neutral     3443
positive    3364
negative    2711
Name: count, dtype: int64
Setup complete.


## Model 1 — Logistic Regression + TF-IDF

Grid searches over:
- **TF-IDF** `ngram_range` and `max_features`
- **LR** regularisation strength `C` and `solver`

In [ ]:
# ── Logistic Regression Pipeline ──────────────────────────────────────────────
lr_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', sublinear_tf=True)),
    ('clf',   LogisticRegression(max_iter=1000, random_state=42))
])

lr_param_grid = {
    'tfidf__ngram_range' : [(1, 1), (1, 2)],
    'tfidf__max_features': [10_000, 30_000, None],
    'clf__C'             : [0.1, 1.0, 10.0],
    'clf__solver'        : ['lbfgs', 'saga'],
}

lr_gs = GridSearchCV(
    lr_pipeline, lr_param_grid,
    cv=cv, scoring='f1_macro',
    n_jobs=-1, verbose=1
)

lr_gs.fit(X_train, y_train)

# Save best model
lr_pkl = '../data/pkls/logistic_regression.pkl'
joblib.dump(lr_gs.best_estimator_, lr_pkl)

evaluate_and_store("Logistic Regression", lr_gs.best_estimator_, lr_gs.best_params_)


Fitting 5 folds for each of 36 candidates, totalling 180 fits

  Logistic Regression
  Best params : {'clf__C': 10.0, 'clf__solver': 'lbfgs', 'tfidf__max_features': None, 'tfidf__ngram_range': (1, 2)}
  Accuracy    : 0.7841
  Macro F1    : 0.7823

Classification Report:
              precision    recall  f1-score   support

    negative       0.81      0.72      0.76       542
     neutral       0.81      0.80      0.80       689
    positive       0.75      0.82      0.78       673

    accuracy                           0.78      1904
   macro avg       0.79      0.78      0.78      1904
weighted avg       0.79      0.78      0.78      1904



## Model 2 — Multinomial Naive Bayes + TF-IDF

Grid searches over:
- **TF-IDF** `ngram_range` and `max_features`
- **MNB** smoothing parameter `alpha`

> **Note:** MNB requires non-negative feature values; TF-IDF with `use_idf=True` can produce negative results for some configs, so we use `sublinear_tf=False` here to guarantee non-negative inputs.

In [8]:
# ── Multinomial Naive Bayes Pipeline ──────────────────────────────────────────
mnb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', sublinear_tf=False)),
    ('clf',   MultinomialNB())
])

mnb_param_grid = {
    'tfidf__ngram_range' : [(1, 1), (1, 2)],
    'tfidf__max_features': [10_000, 30_000, None],
    'clf__alpha'         : [0.01, 0.1, 0.5, 1.0],
}

mnb_gs = GridSearchCV(
    mnb_pipeline, mnb_param_grid,
    cv=cv, scoring='f1_macro',
    n_jobs=-1, verbose=1
)
mnb_gs.fit(X_train, y_train)

# Save best model
mnb_pkl = '../data/pkls/naive_bayes.pkl'
joblib.dump(mnb_gs.best_estimator_, mnb_pkl)

evaluate_and_store("Multinomial Naive Bayes", mnb_gs.best_estimator_, mnb_gs.best_params_)


Fitting 5 folds for each of 24 candidates, totalling 120 fits

  Multinomial Naive Bayes
  Best params : {'clf__alpha': 0.5, 'tfidf__max_features': 10000, 'tfidf__ngram_range': (1, 2)}
  Accuracy    : 0.7547
  Macro F1    : 0.7505

Classification Report:
              precision    recall  f1-score   support

    negative       0.78      0.65      0.71       542
     neutral       0.78      0.80      0.79       689
    positive       0.71      0.79      0.75       673

    accuracy                           0.75      1904
   macro avg       0.76      0.75      0.75      1904
weighted avg       0.76      0.75      0.75      1904



## Model 3 — Random Forest Classifier + TF-IDF

Grid searches over:
- **TF-IDF** `min_df` and `sublinear_tf` are kept at defaults
- **RF** `n_estimators`, `max_features`, `max_depth`, `min_samples_split`, `min_samples_leaf`, `bootstrap`

Random Forest is an ensemble of decision trees trained on bootstrapped subsets of the data. It is less suited to raw TF-IDF text (due to high dimensionality) but serves as a strong non-linear baseline and gives us useful feature importance signals.

In [17]:
# ── Random Forest Pipeline ──────────────────────────────────────────────
rfc_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', sublinear_tf=False)),
    ('clf',   RandomForestClassifier())
])

rfc_param_grid = {
    'clf__n_estimators': [100, 150, 200],
    'clf__max_features': ['sqrt', 'log2'],
    'clf__max_depth': [10, 30, 50],
    'clf__min_samples_split': [5, 10],
    'clf__min_samples_leaf': [2, 4],
    'clf__bootstrap': [True, False]
}

rfc_gs = GridSearchCV(
    rfc_pipeline, rfc_param_grid,
    cv=3, scoring='f1_macro',
    n_jobs=-1, verbose=2
)
rfc_gs.fit(X_train, y_train)

# Save best model
rfc_pkl = '../data/pkls/random_forest.pkl'
joblib.dump(rfc_gs.best_estimator_, rfc_pkl)

evaluate_and_store("Random Forest Classifier", rfc_gs.best_estimator_, rfc_gs.best_params_)

Fitting 3 folds for each of 144 candidates, totalling 432 fits

  Random Forest Classifier
  Best params : {'clf__bootstrap': False, 'clf__max_depth': 50, 'clf__max_features': 'sqrt', 'clf__min_samples_leaf': 2, 'clf__min_samples_split': 10, 'clf__n_estimators': 200}
  Accuracy    : 0.7468
  Macro F1    : 0.7410

Classification Report:
              precision    recall  f1-score   support

    negative       0.87      0.59      0.70       542
     neutral       0.69      0.85      0.76       689
    positive       0.75      0.77      0.76       673

    accuracy                           0.75      1904
   macro avg       0.77      0.74      0.74      1904
weighted avg       0.76      0.75      0.74      1904



## Model 4 — Linear SVM + TF-IDF

Grid searches over:
- **TF-IDF** `ngram_range` and `min_df`
- **LinearSVC** `C`, `loss` function, and `class_weight`

Linear SVM is one of the best-performing classical algorithms for high-dimensional text data. It finds a hyperplane that maximises the margin between classes in TF-IDF feature space. The `class_weight='balanced'` option handles the slight class imbalance in our dataset automatically.

In [ ]:
svm_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', sublinear_tf=True)),
    ('clf', LinearSVC(random_state=42, max_iter=2000))
])


svm_param_grid = {
    'tfidf__ngram_range': [(1,2), (1,3)],
    'tfidf__max_features': [30000, None],
    'tfidf__min_df': [2, 5],
    'tfidf__max_df': [0.9],
    'clf__C': [1, 10, 100],
    'clf__class_weight': ['balanced']
}


svm_gs = GridSearchCV(
    svm_pipeline,
    svm_param_grid,
    cv=3,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=2
)

svm_gs.fit(X_train, y_train)


best_model = svm_gs.best_estimator_

print("\n Best Parameters:")
print(svm_gs.best_params_)


joblib.dump(best_model, '../data/pkls/svm.pkl')

print("\nModel saved as svm.pkl")

from sklearn.metrics import classification_report, accuracy_score

y_pred = best_model.predict(X_test)

print("\n Accuracy:", accuracy_score(y_test, y_pred))
print("\n Classification Report:\n")
print(classification_report(y_test, y_pred))



Fitting 3 folds for each of 24 candidates, totalling 72 fits

✅ Best Parameters:
{'clf__C': 1, 'clf__class_weight': 'balanced', 'tfidf__max_df': 0.9, 'tfidf__max_features': 30000, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2)}

💾 Model saved as svm.pkl

📊 Accuracy: 0.7951680672268907

📄 Classification Report:

              precision    recall  f1-score   support

    negative       0.80      0.73      0.77       542
     neutral       0.80      0.81      0.81       689
    positive       0.79      0.83      0.81       673

    accuracy                           0.80      1904
   macro avg       0.80      0.79      0.79      1904
weighted avg       0.80      0.80      0.79      1904



## Model 5 — XGBoost + TF-IDF

Grid searches over:
- **TF-IDF** `ngram_range` and `max_features`
- **XGBoost** `n_estimators`, `max_depth`, `learning_rate`, `subsample`, `colsample_bytree`

XGBoost (eXtreme Gradient Boosting) is a gradient-boosted tree ensemble that builds trees sequentially, with each tree correcting the errors of the previous one. TF-IDF features are first truncated via **TruncatedSVD** (LSA) to a dense lower-dimensional representation, since XGBoost does not natively handle sparse matrices as efficiently as linear models.

> **Note:** We apply TruncatedSVD (`n_components=300`) as a dimensionality reduction step between TF-IDF and XGBoost in the pipeline to keep training time manageable.

In [30]:
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)

xgb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', sublinear_tf=True)),
    ('svd', TruncatedSVD(n_components=300, random_state=42)),
    ('clf', XGBClassifier(
        objective='multi:softmax',
        num_class=3,
        eval_metric='mlogloss',
        random_state=42,
        n_jobs=-1
    ))
])

xgb_param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__max_features': [8000, 12000],
    'tfidf__min_df': [3, 5],
    'clf__n_estimators': [200, 400],
    'clf__max_depth': [3, 4, 5],
    'clf__learning_rate': [0.01, 0.05, 0.1],
    'clf__subsample': [0.8],
    'clf__colsample_bytree': [0.8]
}

xgb_gs = GridSearchCV(
    xgb_pipeline, xgb_param_grid,
    cv=3, scoring='f1_macro',
    n_jobs=-1, verbose=2
)
xgb_gs.fit(X_train, y_train_enc)

xgb_pkl = '../data/pkls/xgboost.pkl'
joblib.dump(xgb_gs.best_estimator_, xgb_pkl)

evaluate_and_store("XGBoost", xgb_gs.best_estimator_, xgb_gs.best_params_)

Fitting 3 folds for each of 144 candidates, totalling 432 fits


ValueError: Mix of label input types (string and number)

In [31]:
y_pred_enc = xgb_gs.best_estimator_.predict(X_test)
y_pred_str = le.inverse_transform(y_pred_enc)

report = classification_report(y_test, y_pred_str)
entry = {
    "model"                : "XGBoost",
    "best_params"          : xgb_gs.best_params_,
    "accuracy"             : accuracy_score(y_test, y_pred_str),
    "macro_f1"             : f1_score(y_test, y_pred_str, average='macro'),
    "weighted_f1"          : f1_score(y_test, y_pred_str, average='weighted'),
    "macro_precision"      : precision_score(y_test, y_pred_str, average='macro'),
    "macro_recall"         : recall_score(y_test, y_pred_str, average='macro'),
    "classification_report": report,
    "confusion_matrix"     : confusion_matrix(y_test, y_pred_str),
}
model_results.append(entry)

print(f"\n{'='*60}")
print(f"  XGBoost")
print(f"{'='*60}")
print(f"  Best params : {xgb_gs.best_params_}")
print(f"  Accuracy    : {entry['accuracy']:.4f}")
print(f"  Macro F1    : {entry['macro_f1']:.4f}")
print("\nClassification Report:")
print(report)



  XGBoost
  Best params : {'clf__colsample_bytree': 0.8, 'clf__learning_rate': 0.05, 'clf__max_depth': 4, 'clf__n_estimators': 400, 'clf__subsample': 0.8, 'tfidf__max_features': 8000, 'tfidf__min_df': 3, 'tfidf__ngram_range': (1, 1)}
  Accuracy    : 0.7232
  Macro F1    : 0.7164

Classification Report:
              precision    recall  f1-score   support

    negative       0.76      0.59      0.66       542
     neutral       0.73      0.79      0.76       689
    positive       0.70      0.77      0.73       673

    accuracy                           0.72      1904
   macro avg       0.73      0.71      0.72      1904
weighted avg       0.73      0.72      0.72      1904



## Results Summary

Aggregated view of all trained models so far.

In [32]:
# ── Summary table ─────────────────────────────────────────────────────────────
summary_df = pd.DataFrame([{
    "Model"          : r["model"],
    "Accuracy"       : round(r["accuracy"],        4),
    "Macro F1"       : round(r["macro_f1"],        4),
    "Weighted F1"    : round(r["weighted_f1"],     4),
    "Macro Precision": round(r["macro_precision"], 4),
    "Macro Recall"   : round(r["macro_recall"],    4),
    "Best Params"    : r["best_params"],
} for r in model_results])

display(summary_df)

,Model,Accuracy,Macro F1,Weighted F1,Macro Precision,Macro Recall,Best Params
0,Logistic Regression,0.7841,0.7823,0.7839,0.7873,0.7799,"{'clf__C': 10.0, 'clf__solver': 'lbfgs', 'tfid..."
1,Multinomial Naive Bayes,0.7547,0.7505,0.7537,0.7581,0.7478,"{'clf__alpha': 0.5, 'tfidf__max_features': 100..."
2,Random Forest Classifier,0.7468,0.7410,0.7440,0.7705,0.7356,"{'clf__bootstrap': False, 'clf__max_depth': 50..."
3,SVM,0.7920,0.7898,0.7917,0.7936,0.7881,"{'clf__C': 1, 'clf__class_weight': 'balanced',..."
4,XGBoost,0.7232,0.7164,0.7206,0.7279,0.7139,"{'clf__colsample_bytree': 0.8, 'clf__learning_..."


### Results at a Glance

| Rank | Model | Macro F1 | Accuracy |
|------|-------|----------|----------|
| 1 | **Linear SVM** | **0.7898** | **0.7920** |
| 2 | Logistic Regression | 0.7823 | 0.7841 |
| 3 | Multinomial Naive Bayes | 0.7505 | 0.7547 |
| 4 | Random Forest | 0.7410 | 0.7468 |
| 5 | XGBoost | 0.7164 | 0.7232 |

### Observations

1. **Linear models dominate**: SVM and Logistic Regression are the top two, which is expected for TF-IDF features in high-dimensional space. These models excel at finding separating hyperplanes in sparse, high-dimensional data.

2. **Tree-based models underperform**: Random Forest and XGBoost struggle with raw TF-IDF vectors — the feature space is too wide and sparse for tree splits to be effective. Even with TruncatedSVD, XGBoost loses information in the dimensionality reduction step.

3. **Class imbalance is mild**: With ~3443 neutral / 3364 positive / 2711 negative samples, the class distribution is fairly balanced. The `negative` class consistently has the lowest recall across all models, suggesting those sentences are harder to classify (possibly because negative financial language is more hedged and indirect).

4. **Bigrams help**: All top models selected `ngram_range=(1,2)`, indicating that 2-word phrases like "net loss", "strong growth", "decline in" carry critical sentiment signals beyond individual words.

5. **All baselines plateau around ~0.79**: The convergence across different algorithms suggests we are hitting the **ceiling of what bag-of-words representations can capture**. TF-IDF does not understand word order, context, negation scope, or semantic meaning — it just counts (weighted) words.

### Why We Can't Break 0.80 with TF-IDF

The remaining ~20% of errors come from cases that require **deeper language understanding**:
- **Negation**: *"not a significant risk"* → positive, but TF-IDF sees "risk" as negative
- **Hedging / Pragmatics**: *"we may experience challenges"* → the word "may" changes everything
- **Context / Discourse**: *"revenue grew 15%"* is positive alone, but negative if preceded by *"despite expectations of 30% growth"*
- **Implicit sentiment**: *"the company is exploring strategic alternatives"* → sounds neutral but is almost always negative (distress signal)

These require **contextual embeddings** (transformers) rather than word-counting approaches.

All baseline models have been saved to `../data/pkls/` and can be loaded for comparison at any time."""